# Track A — Random Forest (Karan)

Hourly heat demand per building type (`profile_type`). Data: `data/modeling/hourly_heat.parquet`
(17 profile types x 8,760 hours of 2018). Train Jan-Sep, predict Oct-Dec, score against
the baseline (0.392 mean CV(RMSE)) with `scripts/score.py`.

Features: weather + calendar + `profile_type` only - no past demand. Target: `heat_intensity`,
converted back to kWh with `* scale_value` before scoring.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("couldn't find repo root (pyproject.toml)")


ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

from heatreuse import io, config, timeutils, units, metrics
from heatreuse.forecasting.splits import month_folds

pd.set_option("display.width", 120)

# reset_index so positions line up exactly with month_folds' own internal
# ts.reset_index(drop=True) - every array below (X_all, y_all, ...) stays in this order.
hourly = pd.read_parquet(ROOT / "data" / "modeling" / "hourly_heat.parquet").reset_index(drop=True)
print(hourly.shape)
print(hourly["data_split"].value_counts())
hourly.head()

## 2. Quick EDA — profile types and the heat-vs-temperature shape

`low_model_count` flags the types built from few simulated buildings (noisier signal) —
worth knowing before the diagnostics section later.

In [ ]:
profile_info = (hourly.groupby("profile_type", observed=True)
                .agg(models_used=("models_used", "first"),
                     low_model_count=("low_model_count", "first"),
                     scale_basis=("scale_basis", "first"))
                .sort_values("models_used"))
profile_info

In [ ]:
types = sorted(hourly["profile_type"].unique())
fig, axes = plt.subplots(5, 4, figsize=(16, 16), sharex=True)
for ax, ptype in zip(axes.flat, types):
    g = hourly[hourly["profile_type"] == ptype]
    ax.scatter(g["temp_c"], g["heat_intensity"], s=2, alpha=0.2)
    ax.set_title(ptype, fontsize=8)
for ax in axes.flat[len(types):]:
    ax.axis("off")
fig.suptitle("heat_intensity vs temp_c by profile_type (sanity check: should fall as temp rises)")
fig.tight_layout()
plt.show()

## 3. Features and target

`profile_type` one-hot encoded, not ordinal: `RandomForestRegressor` has no native
categorical support, and ordinal encoding would impose a fake order on 17 unrelated
building types. One-hot is fit on the whole table once up front — the 17 categories are
a fixed, known set, so there is no leakage from doing this before any CV split.

`month` is built separately as a toggle, decided empirically in section 4.

In [ ]:
BASE_FEATURES = ["temp_c", "rel_humidity_pct", "wind_speed_ms", "ghi_wm2", "hdh_18c",
                 "temp_c_roll24", "temp_c_lag1", "temp_c_lag3",
                 "hour_of_day", "day_of_week", "is_weekend", "is_holiday"]
TARGET = "heat_intensity"


def make_design_matrix(df: pd.DataFrame, include_month: bool) -> pd.DataFrame:
    feats = BASE_FEATURES + (["month"] if include_month else [])
    X = df[feats].copy()
    X["is_weekend"] = X["is_weekend"].astype(int)
    X["is_holiday"] = X["is_holiday"].astype(int)
    X["profile_type"] = df["profile_type"]
    return pd.get_dummies(X, columns=["profile_type"])


X_all_with_month = make_design_matrix(hourly, include_month=True)
X_all_no_month = make_design_matrix(hourly, include_month=False)
y_all = hourly[TARGET].to_numpy()

print(X_all_no_month.shape, X_all_with_month.shape)
X_all_no_month.head()

## 4. Month feature: with vs without. OOB vs honest CV (once)

CV harness: one `RandomForestRegressor` per `month_folds` fold (leave-one-month-out
inside Jan-Sep, 168h gap), scored the same way `scripts/score.py` does — CV(RMSE) per
`profile_type`, then the mean. Scoring directly on `heat_intensity` (instead of
converting back to kWh first) gives the identical number here: `cvrmse` divides RMSE by
the mean of the actual values, and `scale_value` is constant within a `profile_type`, so
it cancels out of both the numerator and denominator.

`month` is the only feature whose test-time values (10, 11, 12) never appear in training
(1-9) — a tree can only split on thresholds it has seen, so it would silently extrapolate
every test row into whichever bucket September fell into. This compares with-vs-without
directly rather than assuming that's a problem in practice.

**Note on `n_estimators`:** a single 300-tree fit benchmarked at ~145s on this machine
even with `n_jobs=-1` across 10 cores (a 50-tree fit only dropped to ~27s, so the ceiling
is per-tree cost, not missing parallelism) — the full with/without-month x tuning-grid
search at 300 trees throughout would run for hours. Every CV/search fit below uses
`n_estimators=50`; only the one final refit in section 6 uses the full `n_estimators=300`
as specified, so the actual submission isn't affected.

In [ ]:
def mean_cvrmse_by_type(y, pred, profile_type):
    """Same recipe as scripts/score.py's score_hourly: cvrmse per profile_type, then mean."""
    df = pd.DataFrame({"profile_type": np.asarray(profile_type), "y": y, "pred": pred})
    per_type = {t: metrics.cvrmse(g["y"].to_numpy(), g["pred"].to_numpy())
                for t, g in df.groupby("profile_type", observed=True)}
    s = pd.Series(per_type)
    return s.mean(), s


def cv_scores(X: pd.DataFrame, y: np.ndarray, profile_type: pd.Series, ts: pd.Series,
             rf_kwargs: dict) -> pd.DataFrame:
    fold_rows = []
    for month, trn_idx, val_idx in month_folds(ts):
        rf = RandomForestRegressor(random_state=0, n_jobs=-1, **rf_kwargs)
        rf.fit(X.iloc[trn_idx], y[trn_idx])
        pred = rf.predict(X.iloc[val_idx])
        mean_cv, _ = mean_cvrmse_by_type(y[val_idx], pred, profile_type.iloc[val_idx])
        fold_rows.append({"month": month, "mean_cvrmse": mean_cv})
    return pd.DataFrame(fold_rows)

In [ ]:
# 9 folds x 2 feature sets = 18 fits. n_estimators=50 for this search pass (see note above).
SEARCH_N_ESTIMATORS = 50
scores_with_month = cv_scores(X_all_with_month, y_all, hourly["profile_type"], hourly["timestamp"],
                              rf_kwargs=dict(n_estimators=SEARCH_N_ESTIMATORS))
scores_no_month = cv_scores(X_all_no_month, y_all, hourly["profile_type"], hourly["timestamp"],
                            rf_kwargs=dict(n_estimators=SEARCH_N_ESTIMATORS))

print("with month:", scores_with_month["mean_cvrmse"].mean())
print("no month:  ", scores_no_month["mean_cvrmse"].mean())
pd.DataFrame({"with_month": scores_with_month.set_index("month")["mean_cvrmse"],
             "no_month": scores_no_month.set_index("month")["mean_cvrmse"]})

**Why the raw mean above is misleading:** `cvrmse = RMSE / mean(actual)`. In Jun/Jul/Aug,
several building types have `heat_intensity` near zero (no heating load), so the
denominator is tiny and `cvrmse` explodes (the printed table shows July/August folds in
the hundreds) — that's a numerical artifact, not a real signal about `month`. It also
swamps a plain mean across all 9 folds, which is why the headline numbers above
(23.75 vs 181.35) are far larger than any individual month. Using the **median** fold
score instead of the mean is robust to this; the decision below uses that.

In [ ]:
# Decide on the MEDIAN fold score, not the mean - the mean is dominated by the
# summer cvrmse blowups explained above, which are noise, not signal about `month`.
with_month_median = scores_with_month["mean_cvrmse"].median()
no_month_median = scores_no_month["mean_cvrmse"].median()
print(f"with month, median fold score: {with_month_median:.4f}")
print(f"no month,   median fold score: {no_month_median:.4f}")

USE_MONTH = with_month_median < no_month_median
X_all = X_all_with_month if USE_MONTH else X_all_no_month
print("Using month feature:", USE_MONTH)

### OOB vs honest CV, once

OOB only holds out individual bootstrapped rows. Because consecutive hours are highly
autocorrelated, a left-out row's near-neighbors are almost always still in-bag for some
tree, so OOB understates error on a genuinely unseen block of time — which is exactly
what the real Oct-Dec test is. This is why OOB is never used for model selection below;
shown here once to make the point, not used again.

In [ ]:
train_mask = (hourly["data_split"] == "train").to_numpy()

# Same SEARCH_N_ESTIMATORS as the CV comparison above, so this is an apples-to-apples
# OOB-vs-CV comparison at matched tree count (not a comparison against the final model).
rf_oob = RandomForestRegressor(n_estimators=SEARCH_N_ESTIMATORS, oob_score=True, n_jobs=-1, random_state=0)
rf_oob.fit(X_all.loc[train_mask], y_all[train_mask])

oob_mean_cvrmse, _ = mean_cvrmse_by_type(
    y_all[train_mask], rf_oob.oob_prediction_, hourly.loc[train_mask, "profile_type"])

# Median, not mean, for the same reason as the month decision above - the mean is
# dominated by the summer cvrmse blowups, which would wildly overstate how much more
# optimistic OOB is.
true_cv_median = (scores_with_month if USE_MONTH else scores_no_month)["mean_cvrmse"].median()
print(f"OOB mean CV(RMSE):            {oob_mean_cvrmse:.4f}")
print(f"Honest month-fold median CV(RMSE): {true_cv_median:.4f}")

## 5. Tuning grid (`n_estimators=50` for search — see the note in section 4; the final
## submission in section 6 refits at the full `n_estimators=300`)

`max_features x min_samples_leaf`, 9 combinations, averaged over the 9 `month_folds`
folds, picked by mean CV(RMSE) (never OOB). `max_features=1.0` with row bootstrapping is
plain bagging; `max_features<1.0` adds the random-subspace trick that turns bagging into
a proper Random Forest — this grid is a direct, presentable answer to how much that buys
over bagging alone.

If this cell is still too slow on your machine, add `max_samples=0.5` to `kwargs` below
and re-run — not included by default, only as a fallback.

In [ ]:
import itertools
import time

GRID = list(itertools.product([0.33, 0.5, 1.0], [1, 5, 20]))  # (max_features, min_samples_leaf)
grid_results = []
t0 = time.time()
for max_features, min_samples_leaf in GRID:
    kwargs = dict(n_estimators=SEARCH_N_ESTIMATORS, max_features=max_features,
                 min_samples_leaf=min_samples_leaf, max_depth=None)
    folds = cv_scores(X_all, y_all, hourly["profile_type"], hourly["timestamp"], rf_kwargs=kwargs)
    grid_results.append({"max_features": max_features, "min_samples_leaf": min_samples_leaf,
                         "mean_cvrmse": folds["mean_cvrmse"].mean(),
                         "jan_cvrmse": folds.loc[folds["month"] == 1, "mean_cvrmse"].iloc[0],
                         "feb_cvrmse": folds.loc[folds["month"] == 2, "mean_cvrmse"].iloc[0]})
    print(grid_results[-1], f"  ({time.time() - t0:.0f}s elapsed)")

grid_df = pd.DataFrame(grid_results).sort_values("mean_cvrmse").reset_index(drop=True)
grid_df

## 6. Final fit on all Jan–Sep, predict Oct–Dec once

Refit with the winning grid settings on every training row, predict the test rows
exactly once, convert back to kWh, and clip negatives to 0 — mirroring the same
`np.clip(..., 0, None)` pattern `scripts/baselines.py` already uses.

In [ ]:
best = grid_df.iloc[0]
BEST_KWARGS = dict(n_estimators=300, max_features=best["max_features"],
                   min_samples_leaf=int(best["min_samples_leaf"]), max_depth=None)
print("Best hyperparameters:", BEST_KWARGS, " USE_MONTH =", USE_MONTH)

test_mask = (hourly["data_split"] == "test").to_numpy()

final_rf = RandomForestRegressor(random_state=0, n_jobs=-1, **BEST_KWARGS)
final_rf.fit(X_all.loc[train_mask], y_all[train_mask])

pred_intensity = final_rf.predict(X_all.loc[test_mask])
predicted_kwh = np.clip(pred_intensity * hourly.loc[test_mask, "scale_value"].to_numpy(), 0, None)

submission = pd.DataFrame({
    "timestamp": hourly.loc[test_mask, "timestamp"].to_numpy(),
    "profile_type": hourly.loc[test_mask, "profile_type"].to_numpy(),
    "predicted_kwh": predicted_kwh,
})
submission.shape

## 7. Pre-submission checks

Replicates `scripts/score.py`'s own merge check before running it for real, so a bad
submission file is caught here with a clear message rather than via its `SystemExit`.

In [ ]:
assert len(submission) == 37536, len(submission)
assert submission["predicted_kwh"].isna().sum() == 0
assert (submission["predicted_kwh"] < 0).sum() == 0
assert not submission.duplicated(["timestamp", "profile_type"]).any()
assert set(submission["profile_type"]) == set(hourly["profile_type"].unique())

truth_test = hourly.loc[test_mask, ["timestamp", "profile_type"]]
check = truth_test.merge(submission, on=["timestamp", "profile_type"], how="outer", indicator=True)
assert (check["_merge"] == "both").all(), check["_merge"].value_counts()

print("all checks passed:", len(submission), "rows")

## 8. Write submission and score it

Scores via the real `scripts/score.py` subprocess rather than reimplementing its logic —
it's the one shared path everyone's score goes through, and it's what appends to
`submissions/leaderboard.csv`. The printed mean CV(RMSE) here, not any CV number above,
is what's comparable to the 0.392 baseline.

In [ ]:
sub_path = ROOT / "submissions" / "karan_hourly.csv"
submission.to_csv(sub_path, index=False)
print("wrote", sub_path)

In [ ]:
import subprocess

result = subprocess.run(["python", "scripts/score.py", "hourly", "submissions/karan_hourly.csv"],
                        cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scoring failed")

## 9. Diagnostics

Per-type `cvrmse`/`nmbe`/`wape` is already printed by `score.py` above — tie the worst
types back to `low_model_count` from section 2.

Permutation importance (not impurity importance) is computed on the **January
validation fold** from `month_folds`, not the final test set. Impurity-based importance
is known to inflate high-cardinality / many-dummy-column features, which matters here
specifically because `profile_type` is one-hot encoded into 17 columns.

In [ ]:
jan_month, jan_trn_idx, jan_val_idx = next(mf for mf in month_folds(hourly["timestamp"]) if mf[0] == 1)

rf_jan = RandomForestRegressor(random_state=0, n_jobs=-1, **BEST_KWARGS)
rf_jan.fit(X_all.iloc[jan_trn_idx], y_all[jan_trn_idx])

perm = permutation_importance(rf_jan, X_all.iloc[jan_val_idx], y_all[jan_val_idx],
                              n_repeats=5, random_state=0, n_jobs=-1)
importances = pd.Series(perm.importances_mean, index=X_all.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
importances.head(15).plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_title("Permutation importance, January validation fold")
fig.tight_layout()
plt.show()

In [ ]:
truth_kwh = hourly.loc[test_mask, ["timestamp", "profile_type", "heat_total_kwh"]]
scored = truth_kwh.merge(submission, on=["timestamp", "profile_type"])
per_type_cvrmse = pd.Series({t: metrics.cvrmse(g["heat_total_kwh"], g["predicted_kwh"])
                            for t, g in scored.groupby("profile_type", observed=True)}).sort_values()

worst_type = per_type_cvrmse.index[-1]
typical_type = per_type_cvrmse.index[len(per_type_cvrmse) // 2]
print("worst:  ", worst_type, per_type_cvrmse.iloc[-1])
print("typical:", typical_type, per_type_cvrmse.loc[typical_type])

for t in [worst_type, typical_type]:
    g = scored[(scored["profile_type"] == t) & (scored["timestamp"].dt.month == 12)].sort_values("timestamp")
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(g["timestamp"], g["heat_total_kwh"], label="actual")
    ax.plot(g["timestamp"], g["predicted_kwh"], label="predicted", alpha=0.8)
    ax.set_title(f"{t} — December actual vs predicted")
    ax.legend()
    fig.tight_layout()
    plt.show()

## 10. Write-up

> Trained a single global `RandomForestRegressor` (scikit-learn, `n_estimators=300`,
> `max_features=0.5`, `min_samples_leaf=5`) on `heat_intensity` across all 17 building
> types, using weather + calendar + one-hot `profile_type`, with `month` included (no
> past demand, which this dataset doesn't include at all). CV showed `month` helped —
> robust (median) leave-one-month-out fold score 0.957 with it vs 1.171 without; a plain
> mean across all 9 folds is misleading here, since `cvrmse` explodes for several
> building types in Jun–Aug when their actual heat is near zero. Tuned with
> leave-one-month-out CV inside Jan–Sep (168h gap); OOB (0.177) was noticeably more
> optimistic than honest CV (0.957, ~5.4x lower), confirming it shouldn't be used for
> model selection — neighbouring hours are near-duplicates, so OOB barely tests
> extrapolation to an unseen month. On the real Oct–Dec test set, `scripts/score.py` gave
> a **mean CV(RMSE) of 0.2787** against the 0.392 baseline — a clear **beat, by 0.113
> (29% relative improvement)**. By permutation importance, the one-hot residential
> `profile_type` columns (`single-family_attached`, `multi-family_with_2_-_4_units`,
> `multi-family_with_5plus_units`) dominate by a wide margin — each outweighs every
> individual weather feature, with `temp_c_lag1` a distant next — confirming building
> type (specifically, residential vs. commercial scale) matters far more than any single
> weather variable. The model is worst on **warehouse** (CV(RMSE) = 0.556) — notably
> *not* a `low_model_count` type (114 simulated buildings backing it), so the error isn't
> from a thin simulated sample; warehouses likely just have more operationally irregular,
> less temperature-driven heating schedules that weather + calendar alone don't capture
> well. The best-performing type was `fullservicerestaurant` (CV(RMSE) = 0.099); a
> representative "typical" type was `outpatient` (CV(RMSE) = 0.290, see the December
> actual-vs-predicted plots above for both).